# Analyse du dédoublonnage

## Fonction d'analyse
Calcul et restitution du nombre de ligne en défut d'intégrité

In [1]:
from datetime import datetime
import json
from tab_dataset import Cdataset
import pandas as pd
import ntv_pandas as npd
import pathlib

def analyse_integrite(data, schema, affiche=True, indic=True):
    '''analyse les relations du DataFrame 'data' définies dans le schéma 'schema'.
    Le nombre de lignes en erreur par relation (dict) est retourné et optionnellement affiché (paramètre 'affiche=True') . 
    Les lignes en erreur sont optionnellement ajoutées (paramètre 'indic=True') à 'data' sous forme de champs booléens par relation.
    '''
    dic_errors = Cdataset(data).check_relationship(schema)
    dic_count = {name: len(errors) for name, errors in dic_errors.items()}
    if affiche:
        for name, total in dic_count.items():
            print('{:<50} {:>5}'.format(name, total))
    if indic:
        data['ok'] = True
        for name, errors in dic_errors.items():
            data[name] = True
            data.loc[errors, name] = False
            data['ok'] = data['ok'] & data[name] 
        if affiche:
            nb_ok = sum(data['ok'])
            nb_ko = len(data) - sum(data['ok'])      
            print("\nnombre d'enregistrements sans erreurs : ", nb_ok)
            print("nombre d'enregistrements avec au moins une erreur : ", nb_ko)
            print("dont doublons : ", dic_count['index - id_pdc_itinerance'])
            print("\ntaux d'erreur : ", round(nb_ko / len(data) * 100), ' %')
    return dic_count

## Schéma de données
Le schéma de données restreint à la propriété 'relationship' et construit à partir du modèle de données est le suivants :

In [2]:
# complément à inclure dans le schéma de données
schema = {
    'relationships': [
         # relation unicité des pdl
         {"fields": ["id_pdc_itinerance", "index"],                    "link" : "coupled" },   
         # relations inter entités
         {"fields": ["id_station_itinerance", "contact_operateur"],    "link" : "derived" },
         {"fields": ["id_station_itinerance", "nom_enseigne"],         "link" : "derived" },
         {"fields": ["id_station_itinerance", "coordonneesXY"],        "link" : "derived" },
         {"fields": ["id_pdc_itinerance", "id_station_itinerance"],    "link" : "derived" },
         # relations intra entité - station
         {"fields": ["id_station_itinerance", "nom_station"],          "link" : "derived" },
         {"fields": ["id_station_itinerance", "implantation_station"], "link" : "derived" },
         #{"fields": ["id_station_itinerance", "date_maj"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "nbre_pdc"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "condition_acces"],      "link" : "derived" },
         {"fields": ["id_station_itinerance", "horaires"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "station_deux_roues"],   "link" : "derived" },
         # relations intra entité - localisation
         {"fields": ["coordonneesXY", "adresse_station"],              "link" : "derived" }
    ]
}

## Initialisation des données
Fichier pandas

In [3]:
origine = 'datagouv_organization_or_owner'
priorite = 'priorite'
coord = 'coordonneesXY'
id_station = 'id_station_itinerance'
id_pdc = 'id_pdc_itinerance'
last_modif = 'last_modified'
date_maj = 'date_maj'
nom_station = 'nom_station'
adresse = 'adresse_station'
amenageur = 'nom_amenageur'
unit = 'unite'

unicite_stations = [id_station, origine, date_maj, last_modif]
filtre = [priorite,  date_maj, last_modif]
id_station_pdc = [id_station, id_pdc]
att_station = [id_station, date_maj, last_modif, amenageur, nom_station, coord]
att_pdc = [id_pdc, id_station, date_maj, last_modif, amenageur, nom_station, coord, adresse, origine]
filtre_qualicharge = [nom_station, adresse, coord]

Consolidation statique dédoublonnée : https://proxy.transport.data.gouv.fr/resource/consolidation-transport-irve-statique
Consolidation avec doublons : https://proxy.transport.data.gouv.fr/resource/consolidation-transport-avec-doublons-irve-statique

In [4]:
file_irve_brut = 'consolidation_transport_avec_doublons_irve_statique_14_03.csv' # données brutes
file_irve = 'consolidation_transport_irve_statique_14_03.csv' # données dédoublonnées

irve_brut = pd.read_csv(file_irve_brut, sep=',', low_memory=False, dtype='object').reset_index()
irve_brut[last_modif] = irve_brut['datagouv_last_modified']

irve = pd.read_csv(file_irve, sep=',', low_memory=False, dtype='object').reset_index()
irve[last_modif] = irve['datagouv_last_modified']

## Données brutes

In [5]:
print('nombre de lignes : {}, nombre de pdc : {} \n'.format(len(irve_brut), len(irve_brut.groupby([id_pdc]).count())))
print(irve_brut.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
res_brut = analyse_integrite(irve_brut, schema)

nombre de lignes : 386786, nombre de pdc : 150001 

datagouv_organization_or_owner
QualiCharge                       61079
Engie Mobilités Electriques       58305
GIREVE                            32363
TotalEnergies Marketing France    29490
Mobilize Power Solutions          22029
IZIVIA                            15880
Driveco                           15511
ubitricity                        15349
Alizé                             14501
STATIONS-E                        14344
Name: index, dtype: int64 

index - id_pdc_itinerance                          300276
contact_operateur - id_station_itinerance          81791
nom_enseigne - id_station_itinerance               64804
coordonneesXY - id_station_itinerance              119209
id_station_itinerance - id_pdc_itinerance          152690
nom_station - id_station_itinerance                57862
implantation_station - id_station_itinerance       79135
nbre_pdc - id_station_itinerance                   73252
condition_acces - id_station_i

## Test des cas de doublons identifiés

- cas 0 : doublon de pdc d'origine différentes sur une même station
- cas 1 : pdc sur deux stations avec identifiants de station différents
- cas 2 : station avec deux origines, des identifiants différents et mêmes coordonnées
- cas 3 : station avec deux origines, des identifiants et des coordonnées différents, des noms identiques
- cas 4 : station avec deux origines, des identifiants et des coordonnées et des noms différents, des adresses identiques
- cas 5 : station Qualicharge décommissionnée partiellement
- cas 6 : station Qualicharge décommissionnée totalement
- cas 7 : station Qualicharge avec changement d'unité d'exploitation

In [6]:
def test_doublons(irve):
    return {
        'station Tesla de 48 pdc (cas 0)': len(irve[irve[id_station]=='FRTSLP16281'])==48,
        'pdc Atlante sur deux stations (cas 1)': len(irve[irve[id_pdc]=='FRATLE102181'])==1,
        'pdc Ionity sur deux stations (cas 1)': len(irve[irve[id_pdc]=='FRIOYE423408']) == 1,                 
        'stations de 2 pdc (cas 2)': len(irve[irve[coord]=='[-0.00097, 49.32456]'])==2,
        'stations de 3 pdc (cas 2)': len(irve[irve[coord]=='[-0.03405, 48.75304]'])==3,
        'stations de 12 pdc (cas 2)': len(irve[irve[coord]=='[-0.05677, 48.72293]'])==12,
        'stations de 2 pdc (cas 3)': len(irve[irve[nom_station]=='VALENCE EN POITOU_SALLE DES FETES'])==2,
        'stations de 3 pdc (cas 4)': len(irve[irve[adresse]=='15 Av. Président Georges Pompidou'])==3,
        'station R3 de 1 pdc (cas 5)': len(irve[irve[id_station]=='FRR3MP1063577'])==1,
        'station R3 de 3 pdc (cas 6)': len(irve[irve[id_station]=='FRR3MP1030133'])==1,
        'station Total (cas 7)': len(irve[irve[coord]=='[5.40266, 43.26239]'])==2
    }
def cumul_tests(resultat):
    return sum(resultat.values())

## Dédoublonnage actuel

In [7]:
print('nombre de lignes : {} \n'.format(len(irve)))
print(irve.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
print("nombre d'erreur d'intégrité par type de règle :\n")
res_actuel = analyse_integrite(irve, schema)
test_actuel = test_doublons(irve)
print("\nbilan des tests :\n")
test_actuel

nombre de lignes : 149777 

datagouv_organization_or_owner
QualiCharge                                  61079
GIREVE                                       26135
Alizé                                        13847
IZIVIA                                       13009
Indigo Group                                  7273
Driveco                                       3919
Eco-Movement                                  3168
TotalEnergies Marketing France                1849
Engie Mobilités Electriques                   1729
Citeos Ingénierie IdF & Est (Cogelum IdF)     1498
Name: index, dtype: int64 

nombre d'erreur d'intégrité par type de règle :

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance           1823
nom_enseigne - id_station_itinerance                4103
coordonneesXY - id_station_itinerance               5407
id_station_itinerance - id_pdc_itinerance              0
nom_station - id_station_itinerance                 2351
implantation

{'station Tesla de 48 pdc (cas 0)': True,
 'pdc Atlante sur deux stations (cas 1)': True,
 'pdc Ionity sur deux stations (cas 1)': True,
 'stations de 2 pdc (cas 2)': False,
 'stations de 3 pdc (cas 2)': False,
 'stations de 12 pdc (cas 2)': False,
 'stations de 2 pdc (cas 3)': False,
 'stations de 3 pdc (cas 4)': False,
 'station R3 de 1 pdc (cas 5)': False,
 'station R3 de 3 pdc (cas 6)': False,
 'station Total (cas 7)': False}

## Dédoublonnage proposé
Les critères utilisés pour éliminer les doublons sont par ordre : priorite (datagouv_organization_or_owner), date (date_maj, datagouv_last_modified)

In [8]:
# dédoublonnage des stations (suivant critères d'unicité)
stations = irve_brut.drop_duplicates(unicite_stations).copy()

print("nombre de stations brutes {}, stations suivant critère d'unicité {} et stations uniques {} ".format(len(irve_brut), len(stations), len(irve_brut.drop_duplicates(id_station))))

nombre de stations brutes 386786, stations suivant critère d'unicité 129742 et stations uniques 60254 


### Dédoublonnage direct station
A l'issue de cette étape, on a une liste de stations uniques respectant les critères de filtrage

avec l'origine la plus prioritaire et la mise à jour la plus récente

In [9]:
# choix du critère de priorité (Qualicharge)
stations[priorite] = stations[origine] == 'QualiCharge'

# dédoublonnage des stations suivant son id_station_itinerance avec filtrage suivant les critères retenus
stat_direct = stations.sort_values(by=[id_station] + filtre).drop_duplicates(id_station, keep='last').copy()
if stations[priorite].sum() != stat_direct[priorite].sum():
    print('priorité non respectée')

In [10]:
print('nombre de stations initiales {}, stations dédoublonnées {} (supprimées {})'.format(len(stations), len(stat_direct), len(stations) - len(stat_direct)))
stat_direct.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15]

nombre de stations initiales 129742, stations dédoublonnées 60254 (supprimées 69488)


datagouv_organization_or_owner
QualiCharge                                  13684
Eco-Movement                                 10684
GIREVE                                       10069
IZIVIA                                        6677
Alizé                                         4528
GREENEA                                       1910
Load Stations                                 1385
Driveco                                        893
Syndicat Départemental d'Energie du Tarn       700
ZE-WATT                                        623
Citeos Ingénierie IdF & Est (Cogelum IdF)      620
SOREGIES                                       451
Electric 55 Charging                           394
Mobilize Power Solutions                       381
ZEborne                                        377
Name: index, dtype: int64

### Dédoublonnage direct pdc
A l'issue de cette étape, chaque pdc est présent une seule fois sur la station respectant les critères de filtrage

In [11]:
# dédoublonnage des pdc présents sur plusieurs stations
pdc_stat = stat_direct[unicite_stations + [priorite]].merge(irve_brut, how='left', on=unicite_stations)
pdc_stat_unique = pdc_stat.sort_values(by=id_station_pdc + filtre).drop_duplicates(id_station_pdc, keep='last').copy()

# dédoublonnage des pdc suivant son id_pdc avec filtrage par priorite, date_maj, last_modif (priorité déja filtrée sur les stations mais nécessaire)
pdc_direct =  pdc_stat_unique.sort_values(by=[id_pdc] + filtre).drop_duplicates(id_pdc, keep='last').copy()
del(pdc_direct['index'])
pdc_direct = pdc_direct.reset_index()

### Bilan dédoublonnage direct

In [12]:
print("nombre de pdc initial {}, avec dédoublonnage des pdc multi-stations {} et avec dédoublonnage de l'historique {} (supprimés {})\n".format(len(pdc_stat), len(pdc_stat_unique), len(pdc_direct), len(pdc_stat)-len(pdc_direct)))
print(pdc_direct.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
print("nombre d'erreur d'intégrité par type de règle :\n")
res_direct = analyse_integrite(pdc_direct, schema)
propos_direct = test_doublons(pdc_direct)
print("\nbilan des tests :\n")
propos_direct

nombre de pdc initial 176319, avec dédoublonnage des pdc multi-stations 175864 et avec dédoublonnage de l'historique 146848 (supprimés 29471)

datagouv_organization_or_owner
QualiCharge                       61079
GIREVE                            23738
Alizé                             13857
IZIVIA                            12333
Indigo Group                       7273
Eco-Movement                       4081
Driveco                            3355
TotalEnergies Marketing France     1848
Engie Mobilités Electriques        1745
Electric 55 Charging               1300
Name: index, dtype: int64 

nombre d'erreur d'intégrité par type de règle :

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance              2
nom_enseigne - id_station_itinerance                   7
coordonneesXY - id_station_itinerance               1012
id_station_itinerance - id_pdc_itinerance              0
nom_station - id_station_itinerance                 1279
implant

{'station Tesla de 48 pdc (cas 0)': True,
 'pdc Atlante sur deux stations (cas 1)': True,
 'pdc Ionity sur deux stations (cas 1)': True,
 'stations de 2 pdc (cas 2)': False,
 'stations de 3 pdc (cas 2)': False,
 'stations de 12 pdc (cas 2)': False,
 'stations de 2 pdc (cas 3)': False,
 'stations de 3 pdc (cas 4)': False,
 'station R3 de 1 pdc (cas 5)': True,
 'station R3 de 3 pdc (cas 6)': False,
 'station Total (cas 7)': False}

### Dédoublonnage indirect des stations

In [13]:
stations_direct = pdc_direct.drop_duplicates([id_station]).copy()

In [14]:
# Cette fonction donne la liste des stations en éliminant (suivant les critères de filtrage) celles avec le champ 'attribut' identique et venant de plusieurs origines
def indirect_station_ext(stations, attribut, affiche=True):
    dupl_att_origine = ~stations.duplicated(keep=False, subset=[attribut, origine])
    
    stations_ext = stations[dupl_att_origine].copy()
    stations_int = stations[~dupl_att_origine].copy()

    filtrage = stations_ext.sort_values(by=[attribut, priorite])
    stat_att = filtrage.drop_duplicates([attribut], keep='last').copy()
    
    duplicates = filtrage.duplicated(subset=[attribut], keep='last')
    #print(duplicates)
    duplicated = stations_ext.loc[duplicates] #.copy()
    
    resultat = pd.concat([stations_int, stat_att])
    if affiche :
        print("nombre de stations dupliquées pour l'attribut '{:<15}' : {}".format(attribut, len(stations_ext) - len(stat_att)))
        print("nombre initial de stations {}, avec dédoublonnage {} {}\n".format(len(stations), attribut, len(resultat)))
    return (resultat, duplicated)

#### Evaluation des stations avec un attribut identique et origine différente
Cette étape donne la liste des stations en éliminant (suivant les critères de filtrage) celles avec un attribut identiques et venant de plusieurs origines

In [15]:
stations_xy, dupl_xy = indirect_station_ext(stations_direct, coord)
stations_nom, dupl_nom = indirect_station_ext(stations_direct, nom_station)
stations_adr, dupl_adr = indirect_station_ext(stations_direct, adresse)

nombre de stations dupliquées pour l'attribut 'coordonneesXY  ' : 947
nombre initial de stations 45861, avec dédoublonnage coordonneesXY 44914

nombre de stations dupliquées pour l'attribut 'nom_station    ' : 197
nombre initial de stations 45861, avec dédoublonnage nom_station 45664

nombre de stations dupliquées pour l'attribut 'adresse_station' : 111
nombre initial de stations 45861, avec dédoublonnage adresse_station 45750



#### Dédoublonnage des stations hors Qualicharge

In [16]:
stations_xy, dupl_xy = indirect_station_ext(stations_direct, coord)

nombre de stations dupliquées pour l'attribut 'coordonneesXY  ' : 947
nombre initial de stations 45861, avec dédoublonnage coordonneesXY 44914



In [17]:
stations_nom, dupl_nom = indirect_station_ext(stations_xy, nom_station)

nombre de stations dupliquées pour l'attribut 'nom_station    ' : 68
nombre initial de stations 44914, avec dédoublonnage nom_station 44846



In [18]:
stations_adr, dupl_adr = indirect_station_ext(stations_nom, adresse)

nombre de stations dupliquées pour l'attribut 'adresse_station' : 51
nombre initial de stations 44846, avec dédoublonnage adresse_station 44795



In [66]:
# le gain avec la suppression des doublons d'adresse est faible et on supprime quelques stations différentes, on se limite aux coordonnées et au nom
stations_indirect = stations_nom

In [67]:
stations_indirect.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15]

datagouv_organization_or_owner
QualiCharge                                  13684
IZIVIA                                        6633
GIREVE                                        6566
Alizé                                         4243
Eco-Movement                                  3993
Driveco                                        887
ZE-WATT                                        623
Citeos Ingénierie IdF & Est (Cogelum IdF)      619
Syndicat Départemental d'Energie du Tarn       601
SOREGIES                                       396
Electric 55 Charging                           394
ZEborne                                        377
Mobilize Power Solutions                       355
Rossini Energy                                 311
Engie Mobilités Electriques                    309
Name: index, dtype: int64

#### Dédoublonnage des stations Qualicharge
origine Qualicharge, même nom, même coordonnées, même adresse, unité différente

In [68]:
# Cette fonction donne la liste des stations en éliminant (suivant les critères de filtrage) les stations Qualicharge avec les mêmes champs nom, coordonnées et adresse mais des unités différentes
def indirect_qualicharge(stations, affiche=True):
    stat = stations.copy()
    stat[unit] = stat[id_station].str[:5]
    #filtre_qualicharge = [nom_station, adresse, coord]
    
    stat_quali = stat[stat[origine]=='QualiCharge'].copy()
    stat_not_quali = stat[~(stat[origine]=='QualiCharge')].copy()
    
    stat_quali['unique'] = ~stat_quali.duplicated(filtre_qualicharge + [unit])
    filtrage = stat_quali.sort_values(by=['unique'] + filtre_qualicharge + [date_maj])
    stat_quali['doublon'] = filtrage.duplicated(subset=filtre_qualicharge, keep='last')
    
    duplicated = stat_quali[stat_quali['doublon']]
    stat_result = stat_quali[~stat_quali['doublon']]
    stat_result.drop(['doublon', 'unique'], axis=1)

    resultat = pd.concat([stat_not_quali, stat_result])
    resultat.drop(['unite'], axis=1)
    
    if affiche :
        print("nombre de stations dupliquées entre unités : {}".format(len(duplicated)))
        print("nombre initial de stations {}, avec dédoublonnage {}\n".format(len(stations), len(resultat)))
    return (resultat, duplicated)

In [69]:
stat_indirect_quali, dupl_quali = indirect_qualicharge(stations_indirect)

nombre de stations dupliquées entre unités : 1695
nombre initial de stations 44846, avec dédoublonnage 43151



### Dédoublonnage indirect pdc

In [70]:
pdc_indirect = stat_indirect_quali[[id_station]].merge(pdc_direct, how='left', on=id_station)

### Bilan dédoublonnage indirect

In [71]:
print("nombre de pdc initial {} et avec dédoublonnage {} (supprimés {})\n".format(len(pdc_direct), len(pdc_indirect), len(pdc_direct)-len(pdc_indirect)))
print(pdc_indirect.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15], '\n')
print("nombre d'erreur d'intégrité par type de règle :\n")
res_indirect = analyse_integrite(pdc_indirect, schema)
propos_indirect = test_doublons(pdc_indirect)
print("\nbilan des tests :\n")
propos_indirect

nombre de pdc initial 146848 et avec dédoublonnage 139623 (supprimés 7225)

datagouv_organization_or_owner
QualiCharge                                  56711
GIREVE                                       21390
Alizé                                        13832
IZIVIA                                       12333
Indigo Group                                  7263
Eco-Movement                                  4080
Driveco                                       3355
TotalEnergies Marketing France                1845
Engie Mobilités Electriques                   1743
Electric 55 Charging                          1300
Qovoltis                                      1198
Citeos Ingénierie IdF & Est (Cogelum IdF)     1110
Lidl                                          1077
e-Totem                                        896
Rossini Energy                                 855
Name: index, dtype: int64 

nombre d'erreur d'intégrité par type de règle :

index - id_pdc_itinerance                          

{'station Tesla de 48 pdc (cas 0)': True,
 'pdc Atlante sur deux stations (cas 1)': True,
 'pdc Ionity sur deux stations (cas 1)': True,
 'stations de 2 pdc (cas 2)': True,
 'stations de 3 pdc (cas 2)': True,
 'stations de 12 pdc (cas 2)': True,
 'stations de 2 pdc (cas 3)': True,
 'stations de 3 pdc (cas 4)': False,
 'station R3 de 1 pdc (cas 5)': True,
 'station R3 de 3 pdc (cas 6)': False,
 'station Total (cas 7)': True}

## Synthèse des dédoublonnages

In [72]:
print("données brutes       : total pdc {:<7} (nombre d'id_pdc_itinerance différents)".format(len(irve_brut.groupby([id_pdc]).count())))
print('solution actuelle    : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}, nombre de tests réussis {}'.format(len(irve), sum(irve['ok']), len(irve) - sum(irve['ok']), res_actuel['index - id_pdc_itinerance'], sum(test_actuel.values())))
print('proposition direct   : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}, nombre de tests réussis {}'.format(len(pdc_direct), sum(pdc_direct['ok']), len(pdc_direct) - sum(pdc_direct['ok']), res_direct['index - id_pdc_itinerance'], sum(propos_direct.values())))
print('proposition indirect : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}, nombre de tests réussis {}'.format(len(pdc_indirect), sum(pdc_indirect['ok']), len(pdc_indirect) - sum(pdc_indirect['ok']), res_indirect['index - id_pdc_itinerance'], sum(propos_indirect.values())))


données brutes       : total pdc 150001  (nombre d'id_pdc_itinerance différents)
solution actuelle    : total pdc 149777  avec pdc ok 134731 , pdc avec erreur 15046 dont doublons 0, nombre de tests réussis 3
proposition direct   : total pdc 146848  avec pdc ok 136632 , pdc avec erreur 10216 dont doublons 0, nombre de tests réussis 4
proposition indirect : total pdc 139623  avec pdc ok 135432 , pdc avec erreur  4191 dont doublons 0, nombre de tests réussis 9


## Exemples

In [73]:
irve[irve[coord]=='[-0.03405, 48.75304]'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
34772,FREVZEKIEU1,FREVZP4070785939483057665,2025-09-24,2026-02-01T03:01:17.000000+0000,SMEG Développement,EVzen/3B222B60-EC8D-4077-AB1F-1C6D169CB4B1,"[-0.03405, 48.75304]","Rue Maurice Ravel, Argentan 61200 France",GIREVE
34773,FREVZELOMA1,FREVZP4070785939483057665,2025-09-24,2026-02-01T03:01:17.000000+0000,SMEG Développement,EVzen/3B222B60-EC8D-4077-AB1F-1C6D169CB4B1,"[-0.03405, 48.75304]","Rue Maurice Ravel, Argentan 61200 France",GIREVE
34774,FREVZEKIEU2,FREVZP4070785939483057665,2025-09-24,2026-02-01T03:01:17.000000+0000,SMEG Développement,EVzen/3B222B60-EC8D-4077-AB1F-1C6D169CB4B1,"[-0.03405, 48.75304]","Rue Maurice Ravel, Argentan 61200 France",GIREVE
85547,FREVZEFREVZELOMA1,FREVZP3B222B60EC8D4077AB1F1C6D169CB,2026-03-13,2026-03-13T23:59:49.000000+0000,SMEG DEVELOPPEMENT,"EVzen - Argentan, PRIX MIAM Argentan","[-0.03405, 48.75304]",Rue Maurice Ravel 61200 Argentan,QualiCharge
99307,FREVZEFREVZEKIEU1,FREVZP3B222B60EC8D4077AB1F1C6D169CB,2026-03-13,2026-03-13T23:59:49.000000+0000,SMEG DEVELOPPEMENT,"EVzen - Argentan, PRIX MIAM Argentan","[-0.03405, 48.75304]",Rue Maurice Ravel 61200 Argentan,QualiCharge
99360,FREVZEFREVZEKIEU2,FREVZP3B222B60EC8D4077AB1F1C6D169CB,2026-03-13,2026-03-13T23:59:49.000000+0000,SMEG DEVELOPPEMENT,"EVzen - Argentan, PRIX MIAM Argentan","[-0.03405, 48.75304]",Rue Maurice Ravel 61200 Argentan,QualiCharge


In [74]:
irve[irve[coord]=='[5.40266, 43.26239]'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
131793,FRTCBE008442,FRTCBP01910,2025-12-21,2026-03-13T23:59:49.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE",QualiCharge
132023,FRTCBE008441,FRTCBP01910,2025-12-21,2026-03-13T23:59:49.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE",QualiCharge
143609,FRHXWE008441,FRHXWP01910,2026-01-07,2026-03-13T23:59:49.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE",QualiCharge
145445,FRHXWE008442,FRHXWP01910,2026-01-07,2026-03-13T23:59:49.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE",QualiCharge


In [75]:
irve_brut[irve_brut[id_pdc] == 'FRIOYE423402'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
349780,FRIOYE423402,FRIOYP9587247,2026-03-12,2026-03-13T23:59:49.000000+0000,Ionity,IONITY Sorgues,"[4.88964, 44.02441]","Aire de Sorgues, A7, 84700 Sorgues",QualiCharge
377508,FRIOYE423402,FRIOYE423402,2026-03-13,2026-03-13T20:00:21.000000+0000,IONITY,IONITY Sorgues,"[4.88965, 44.02444]","Aire de Sorgues, A7, 84700 Sorgues",Eco-Movement


In [76]:
irve_brut[irve_brut[id_station] == 'FRR3MP1030133'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
129237,FRR3ME5284264,FRR3MP1030133,2026-01-07,2026-01-08T05:09:01.000000+0000,R3,Aulnoy-Lez-Valenciennes - NORAUTO,"[50.33320, 3.51124]","Rue Jules Mousseron, 59300 Aulnoy-lez-Valencie...",R3
129238,FRR3ME5284266,FRR3MP1030133,2026-01-07,2026-01-08T05:09:01.000000+0000,R3,Aulnoy-Lez-Valenciennes - NORAUTO,"[50.33320, 3.51124]","Rue Jules Mousseron, 59300 Aulnoy-lez-Valencie...",R3
129239,FRR3ME5284267,FRR3MP1030133,2026-01-07,2026-01-08T05:09:01.000000+0000,R3,Aulnoy-Lez-Valenciennes - NORAUTO,"[50.33320, 3.51124]","Rue Jules Mousseron, 59300 Aulnoy-lez-Valencie...",R3


In [77]:
irve[irve[id_station]=='FRATLP1136899124034716498'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner


In [78]:
irve_brut[irve_brut[id_station]=='FRATLPFR01092'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
347086,FRATLE102181,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge
347087,FRATLE102172,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge
347508,FRATLE104382,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge
347520,FRATLE102192,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge
347648,FRATLE102171,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge
348623,FRATLE104411,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge
348626,FRATLE104401,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge
348628,FRATLE104392,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge
348639,FRATLE102202,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge
349818,FRATLE102191,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge


In [79]:
irve_brut[irve_brut[adresse]=='15 Av. Président Georges Pompidou'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
182003,FRGSPP10000560051,FRGSPP1000056005,2024-06-03,2024-08-01T08:09:38.000000+0000,CARREFOUR FIGEAC,CARREFOUR FIGEAC,"[2.02451, 44.60613]",15 Av. Président Georges Pompidou,Greenspot (Enersoft)
182004,FRGSPP10000560052,FRGSPP1000056005,2024-06-03,2024-08-01T08:09:38.000000+0000,CARREFOUR FIGEAC,CARREFOUR FIGEAC,"[2.02451, 44.60613]",15 Av. Président Georges Pompidou,Greenspot (Enersoft)
341691,FRGSPE10000560051,FRGSPP89228259,2026-03-13,2026-03-13T23:59:49.000000+0000,Greenspot,Carrefour Market Figeac,"[2.02404, 44.60650]",15 Av. Président Georges Pompidou,QualiCharge
341748,FRGSPE10000560053,FRGSPP89228259,2026-03-13,2026-03-13T23:59:49.000000+0000,Greenspot,Carrefour Market Figeac,"[2.02404, 44.60650]",15 Av. Président Georges Pompidou,QualiCharge
346053,FRGSPE10000560052,FRGSPP89228259,2026-03-13,2026-03-13T23:59:49.000000+0000,Greenspot,Carrefour Market Figeac,"[2.02404, 44.60650]",15 Av. Président Georges Pompidou,QualiCharge


In [80]:
irve[irve[id_pdc]=='FRATLE102181'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
118905,FRATLE102181,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge
